# Onboarding de Nuevos Tipos de Log

**Andrea Olcina** — Trabajo de Fin de Máster

---

Este cuaderno sirve de guía para incorporar nuevos tipos de log a la ejecución. Se añaden varios nuevos tipos de log para poder hacer pruebas y seguir estos pasos.

## 📑 Contenido

1. [Bootstrapping de configuración](#1)
2. [Procesamiento de logs](#2)
3. [Generación de templates y regex](#3)

## 📂 Ejemplos disponibles

Los ejemplos están en la carpeta `sample_logs/`:

| Origen |
|---|
| `linux` |

<a id='1'></a>
## 1️⃣ Bootstrapping de configuración

Utilizaremos el módulo `config_bootstrapper.py` para generar una primera aproximación al fichero de configuración de este tipo de log. Este módulo intenta descubrir los patrones de marcas de tiempo y el nivel de log. Además, ejecuta Drain3 para descubrir los clusters más repetitivos y poder adoptar un patrón.

In [ ]:
!python config/config_bootstrapper.py --origin linux --sample-logs "sample_logs/linux/*.log"

### 🔍 Inspección de clusters descubiertos

In [ ]:
import json

import pandas as pd

with open("sample_logs/linux/top_clusters.json", encoding="utf-8") as f:
    raw_data = json.load(f)

results = pd.DataFrame(raw_data["top_clusters"])
results

> ⚠️ Una vez se ha generado el fichero `<origin>_config.py`, es responsabilidad del desarrollador realizar una revisión manual que complete los campos necesarios para la configuración del origen.

<a id='2'></a>
## 2️⃣ Procesamiento de logs

El siguiente paso es ejecutar el procesamiento de logs sobre estos ficheros nuevos para obtener su información. Como el origen era desconocido, habrán ido a parar a la cola de aprendizaje. Si aún no habían sido procesados, será necesario moverlos a la carpeta de procesamiento de la que se recuperan los ficheros nuevos.

> 💡 **Nota:** si quieres realizar varias iteraciones entre el pipeline de procesamiento y el clustering para detectar correctamente los patrones a extraer, ignorar y agrupar, puedes utilizar una estructura de carpetas fuera de producción. Todas las carpetas que alimentan los algoritmos y sus salidas pueden ser indicadas mediante parámetros en las funciones.

**Ejecutar el pipeline de procesamiento:**

```python
def parsing_execution(log_folder: str = "logs/", ingest_to_opensearch: bool = False,
                       output_folder: str = "parsed_logs",
                       enable_performance_tracking: bool = False,
                       results_output_format: str = "parquet"):
```

**Argumentos principales**

| Argumento | Descripción |
|---|---|
| `log_folder` | Carpeta local donde están los logs a procesar. Solo se usa si `execution_mode == "LOCAL"`; en S3 siempre se parte de la raíz del bucket. |
| `output_folder` | Carpeta donde se guardan los resultados parseados (default: `"parsed_logs"`). |
| `enable_performance_tracking` | Si `True`, mide tiempos por fase/origen y genera `performance_metrics.*` al finalizar. Desactivado por defecto para no añadir overhead en ejecuciones normales. |
| `results_output_format` | Formato de los ficheros de resultados parseados: `'parquet'` (default, eficiente para producción) o `'json'` (más legible, útil para debugging). |

In [ ]:
!python parsing/init.py

<a id='3'></a>
## 3️⃣ Generación de templates y regex

Para poder completar la configuración del origen, necesitamos definir sus patrones a ignorar, extractores y clasificar alguna de sus plantillas. Este proceso es iterativo con ayuda del algoritmo Drain y el clustering aglomerativo.

**Ejecutar Drain3, la extracción de templates, agrupación semántica y conversión a expresiones regulares:**
```bash
python template_generator/orchestrator.py origin
```

**Parámetros principales**

| Argumento | Descripción |
|---|---|
| `origin` | Nombre del origen a procesar (debe existir `config/<origin>_config.py`). |

In [ ]:
!python template_generator/orchestrator.py linux

✅ Los resultados están en la carpeta `templates/`, en el fichero `<origin>_template_regex.json`.